In [ ]:
# Databricks notebook source
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

import os

# ---------------------------------------------------------------------------
# Externalized configuration (Databricks widgets / environment variables).
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "landing")
dbutils.widgets.text("pbi_export_prefix", "")


def _cfg(name: str, default: str = "") -> str:
    """Resolve a parameter: widget -> environment variable -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


CATALOG = _cfg("catalog")
STORAGE_ACCOUNT = _cfg("storage_account")
CONTAINER = _cfg("container", "landing")
PBI_EXPORT_PREFIX = _cfg("pbi_export_prefix")
GOLD_SCHEMA = "gold"

if not CATALOG:
    raise ValueError(
        "Missing required parameter 'catalog': set the catalog that holds "
        "the Gold schema.")
if not STORAGE_ACCOUNT:
    raise ValueError(
        "Missing required parameter 'storage_account': set the storage account "
        "that holds the export container.")
if not PBI_EXPORT_PREFIX:
    raise ValueError(
        "Missing required parameter 'pbi_export_prefix': set gold/pbi_export "
        "or gold/pbi_export_dev; refusing to guess the export target.")


# Report export (Gold -> Parquet + meta.json)

Builds the export consumed by the Power BI report (Import mode): reads the
six Gold tables, projects them to the report contract owned by
`Prepare_PBI_Export` (31 columns for `fact_offers`, including the derived
`GeoCountry`/`GeoRegion`, and the five dimensions) and writes one Parquet
per table plus `meta.json` (always `mode: full`, `audience: pbi`) to ADLS
(`abfss://`).

Runs at the end of the job, after `Gold_Complete_Catalog`, in parallel with
the public web export: the two tasks are independent, so a failure here does
not block the public dashboard. The cluster provides the credential (no SAS
or tokens in the notebook).


In [ ]:
# Gold tables already built: the notebook only projects (no dedup).
fact_offers = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_offers")
fact_offer_skills = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_offer_skills")
dim_skill_list = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_skill_list")
dim_calendar = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_calendar")
dim_companies = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.companies_complete_catalog")
dim_currency = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_currency")


In [ ]:
# Sibling module in the repo (same pattern as Prepare_Gold in gold_build.ipynb).
import Prepare_PBI_Export as pbi_export

dest = pbi_export.export_dest(STORAGE_ACCOUNT, CONTAINER, PBI_EXPORT_PREFIX)

tables = pbi_export.build_pbi_export(
    spark, fact_offers, fact_offer_skills, dim_skill_list, dim_calendar,
    dim_companies, dim_currency)

table_counts, data_date = pbi_export.collect_export_stats(tables)
meta = pbi_export.build_meta(table_counts, data_date)

# size_provider is called inside write_pbi_export after writing the six
# Parquet files and before meta.json. Only .parquet files are counted: on
# re-runs the previous meta.json stays in dest and must not count.
size_provider = lambda: sum(
    f.size for f in dbutils.fs.ls(dest) if f.name.endswith(".parquet"))

# meta is mandatory: with meta=None no meta.json is written.
updated_meta = pbi_export.write_pbi_export(
    spark, tables, dest, meta=meta, size_provider=size_provider)


In [ ]:
size_mb = round(updated_meta["size_bytes"] / 1024 / 1024, 2)

print("Report export: summary")
print(f"  destination: {dest}")
print(f"  audience:    {updated_meta['audience']}")
print(f"  mode:        {updated_meta['mode']}")
print(f"  size:        {updated_meta['size_bytes']} bytes ({size_mb} MB)")
print(f"  data date:   {updated_meta['data_date']}")
print(f"  generated:   {updated_meta['generated_at']}")
print(f"  tables:      {updated_meta['tables']}")

display(spark.createDataFrame(
    [("dest", dest),
     ("audience", updated_meta["audience"]),
     ("mode", updated_meta["mode"]),
     ("size_bytes", str(updated_meta["size_bytes"])),
     ("size_mb", str(size_mb)),
     ("data_date", str(updated_meta["data_date"])),
     ("generated_at", updated_meta["generated_at"])],
    ["field", "value"]))
